# RAG Pipeline Tutorial — From Scratch with SevaForge

## What is RAG (Retrieval-Augmented Generation)?

Large Language Models (LLMs) like Claude, GPT-4, and Gemini are powerful, but they have a critical weakness: **they hallucinate.** They confidently generate text that *sounds* right but may be factually wrong, outdated, or completely fabricated. They only know what was in their training data — they can't access your company's docs, your codebase, or any information created after their training cutoff.

**RAG (Retrieval-Augmented Generation)** solves this by giving the LLM an "open book" to reference before answering. Instead of relying solely on memorized knowledge, the model first *retrieves* relevant documents from your knowledge base, then *augments* its prompt with that context before *generating* an answer.

Think of it like the difference between:
- **Closed-book exam** (vanilla LLM): Answer from memory only. Prone to errors on details.
- **Open-book exam** (RAG): Look up the relevant textbook pages first, then answer. Grounded in facts.

### The 5-Stage RAG Pipeline

```
INGESTION (offline, batch — done once)
───────────────────────────────────────
1. LOAD    → Read files, extract metadata (source, type, structure)
2. CHUNK   → Split documents into bite-sized pieces (200-600 tokens each)
3. EMBED   → Convert each chunk into a numerical vector (text → [0.12, -0.34, ...])
4. STORE   → Save vectors in a searchable database

RETRIEVAL (online, per-query — every time a user asks)
───────────────────────────────────────────────────────
5. RETRIEVE → Embed the query, find similar vectors, return matching chunks
              Then pass those chunks as context to an LLM for answer generation.
```

This notebook walks through every stage hands-on using the SevaForge RAG pipeline, which is built from first principles using only `numpy` and Python's standard library. No black-box frameworks — you'll understand exactly what LangChain and ChromaDB do under the hood.

---
## 1. Setup

First, let's import the SevaForge RAG pipeline modules. The entire pipeline is organized into clean, single-responsibility modules:

| Module | Purpose |
|---|---|
| `document_loader` | Read files and extract metadata |
| `chunking` | Split documents into retrieval-friendly pieces |
| `embeddings` | Convert text into numerical vectors |
| `vector_store` | Store and search vectors by similarity |
| `retriever` | Orchestrate retrieval strategies |
| `pipeline` | Wire everything into one interface |
| `evaluation` | Measure retrieval quality with IR metrics |

In [ ]:
import sys
sys.path.insert(0, "../src")

from sevaforge.rag import (
    # Document Loading
    Document, DocumentLoader,
    # Chunking
    Chunk, FixedSizeChunker, SemanticChunker, CodeAwareChunker, HybridChunker,
    # Embeddings
    HashEmbedder, TFIDFEmbedder, CosineDistance, EuclideanDistance,
    # Vector Store
    VectorRecord, SearchResult, InMemoryVectorStore,
    # Retrieval
    SemanticRetriever, KeywordRetriever, HybridRetriever, Reranker,
    # Pipeline
    RAGConfig, RAGPipeline,
    # Evaluation
    EvalCase, RAGEvaluator,
)
import numpy as np

print("All SevaForge RAG modules loaded successfully!")
print(f"numpy version: {np.__version__}")

---
## 2. Document Loading — The First Step

Before you can search through documents, you need to **load them** into a standard format. The `DocumentLoader` reads files from disk and extracts useful metadata that helps downstream stages.

### Why metadata matters

When you retrieve a chunk later, you need to know:
- **Where** it came from (file path, section heading)
- **What kind** of content it is (code, documentation, config)
- **What structure** it has (classes, functions, headers)

This metadata isn't just bookkeeping — it's a first-class signal for retrieval. A query about "authentication" should prefer chunks from `auth.py` over chunks that merely mention the word "authentication" in a README.

Let's load some real files from the SevaForge codebase:

In [ ]:
loader = DocumentLoader()

# Load a Python file — notice the rich structural metadata extracted
py_doc = loader.load_file("../src/sevaforge/rag/embeddings.py")
print(f"=== Python Document ===")
print(f"  Source:      {py_doc.source}")
print(f"  Type:        {py_doc.doc_type}")
print(f"  Token est.:  ~{py_doc.token_estimate} tokens")
print(f"  Classes:     {py_doc.metadata.get('classes', [])}")
print(f"  Functions:   {py_doc.metadata.get('functions', [])[:8]}...")
print(f"  Imports:     {py_doc.metadata.get('imports', [])[:6]}...")
print()

# Load from raw text — useful for API/database content
text_doc = loader.load_text(
    content="RAG combines retrieval with generation to ground LLM answers in real data. "
            "It prevents hallucination by fetching relevant documents before answering.",
    metadata={"topic": "RAG", "author": "tutorial"},
    source="knowledge-base"
)
print(f"=== Text Document ===")
print(f"  Source:  {text_doc.source}")
print(f"  Type:    {text_doc.doc_type}")
print(f"  ID:      {text_doc.id}")
print(f"  Content: {text_doc.content[:80]}...")

---
## 3. Chunking Strategies — The Most Critical Decision

Chunking is arguably the **most impactful** decision in a RAG pipeline. Get it wrong and your retrieval will fail no matter how good your embeddings or vector store are.

### Why we chunk

Embedding models have limited context windows. You can't embed a 10,000-line file as a single vector — the meaning gets diluted into a vague average. Instead, you split documents into focused chunks where each chunk is about **one idea or concept**.

### The key tension

| Chunk Size | Problem |
|---|---|
| Too small (50 tokens) | Loses context. A function signature without its body is useless. |
| Too large (2000 tokens) | Dilutes meaning. If a chunk covers 5 topics, it weakly matches all of them. |
| Sweet spot (200-600 tokens) | Each chunk captures one complete thought with enough context. |

### Why overlap between chunks?

Imagine a paragraph split at the 512-token boundary. The first chunk ends with "The authentication system uses" and the next starts with "JWT tokens with RS256 signing." Neither chunk captures the complete thought. By overlapping 50-100 tokens, the boundary region appears in both chunks, so at least one has the complete sentence.

Let's see the three strategies in action:

In [ ]:
# === Strategy 1: Fixed-Size Chunking ===
# The simplest approach: slide a fixed-size window over the text with overlap.
# Good for unstructured text where there's no natural boundary.

sample_text = """Retrieval-Augmented Generation (RAG) is a technique that enhances
large language model responses by retrieving relevant documents from a knowledge
base before generating an answer. The key insight is that LLMs are powerful
generators but unreliable knowledge stores. By providing relevant context at
inference time, RAG dramatically reduces hallucination.

The RAG pipeline consists of two phases: ingestion and retrieval. During
ingestion, documents are loaded, chunked into smaller pieces, embedded into
dense vectors, and stored in a vector database. During retrieval, the user's
query is embedded using the same model, and the most similar stored vectors
are retrieved. These retrieved chunks form the context that grounds the LLM's
response in factual information.

Chunking strategy has the biggest impact on retrieval quality. Fixed-size
chunking is the simplest approach — just split at every N tokens. Semantic
chunking splits on natural boundaries like paragraph breaks and headings.
Code-aware chunking splits at function and class boundaries to keep complete
logical units together."""

doc = loader.load_text(sample_text, source="rag-overview")

fixed_chunker = FixedSizeChunker(chunk_size=80, overlap=15)
fixed_chunks = fixed_chunker.chunk(doc)

print(f"=== Fixed-Size Chunking (80 tokens, 15 overlap) ===")
print(f"Input: ~{doc.token_estimate} tokens → {len(fixed_chunks)} chunks\n")

for i, chunk in enumerate(fixed_chunks):
    preview = chunk.content[:120].replace('\n', ' ')
    print(f"  Chunk {i}: {chunk.token_count} tokens — \"{preview}...\"")

print(f"\nNotice: chunk boundaries are arbitrary — they can split mid-sentence!")

In [ ]:
# === Strategy 2: Semantic Chunking ===
# Splits on natural boundaries: paragraph breaks, headings, blank lines.
# Produces variable-size chunks that respect the author's structure.

markdown_text = """# RAG Pipeline Overview

RAG (Retrieval-Augmented Generation) is the industry standard for grounding LLM
responses in factual data. It works by retrieving relevant documents before
generating an answer.

## How It Works

The pipeline has two phases: ingestion and retrieval. During ingestion,
documents are chunked, embedded, and stored. During retrieval, a query
is matched against stored vectors.

## Why It Matters

Without RAG, LLMs hallucinate. They generate plausible but incorrect
information. RAG grounds every answer in retrieved evidence, dramatically
improving accuracy and trustworthiness.

## Key Components

The five core components are:
1. Document Loader — reads and normalizes files
2. Chunker — splits documents into retrieval units
3. Embedder — converts text to vectors
4. Vector Store — stores and searches vectors
5. Retriever — orchestrates the search strategy"""

md_doc = Document(id="", content=markdown_text, doc_type="markdown", source="tutorial.md")

semantic_chunker = SemanticChunker(max_chunk_size=200, min_chunk_size=20)
semantic_chunks = semantic_chunker.chunk(md_doc)

print(f"=== Semantic Chunking (markdown) ===")
print(f"Input: {len(markdown_text.split())} words → {len(semantic_chunks)} chunks\n")

for i, chunk in enumerate(semantic_chunks):
    header = chunk.metadata.get('section_header', '(no header)')
    preview = chunk.content[:100].replace('\n', ' ')
    print(f"  Chunk {i}: {chunk.token_count} tokens | Section: '{header}'")
    print(f"           \"{preview}...\"\n")

print("Notice: each chunk respects paragraph/heading boundaries!")

In [ ]:
# === Strategy 3: Code-Aware Chunking ===
# Splits Python code at function/class boundaries.
# CRUCIAL: a function split in half is USELESS for retrieval.

python_code = '''"""Authentication module for the SevaForge platform."""\n
import hashlib
import secrets
from datetime import datetime, timedelta


class TokenManager:
    """Manages JWT token creation and validation."""\n
    def __init__(self, secret_key: str):
        self.secret_key = secret_key
        self.token_expiry = timedelta(hours=1)

    def create_token(self, user_id: str) -> str:
        """Create a new JWT token for the given user."""
        payload = {"user_id": user_id, "exp": datetime.utcnow() + self.token_expiry}
        return self._sign(payload)

    def validate_token(self, token: str) -> dict:
        """Validate a JWT token and return the payload."""
        payload = self._decode(token)
        if payload["exp"] < datetime.utcnow():
            raise ValueError("Token expired")
        return payload


def hash_password(password: str, salt: str = None) -> tuple:
    """Hash a password using SHA-256 with a random salt."""
    if salt is None:
        salt = secrets.token_hex(16)
    hashed = hashlib.sha256(f"{salt}{password}".encode()).hexdigest()
    return hashed, salt


def verify_password(password: str, hashed: str, salt: str) -> bool:
    """Verify a password against its hash."""
    computed, _ = hash_password(password, salt)
    return computed == hashed
'''

code_doc = Document(id="", content=python_code, doc_type="python", source="auth.py")

code_chunker = CodeAwareChunker(max_chunk_size=512)
code_chunks = code_chunker.chunk(code_doc)

print(f"=== Code-Aware Chunking (Python) ===")
print(f"Input: {len(python_code.split(chr(10)))} lines → {len(code_chunks)} chunks\n")

for i, chunk in enumerate(code_chunks):
    block_type = chunk.metadata.get('block_type', '?')
    block_name = chunk.metadata.get('block_name', '?')
    preview = chunk.content[:80].replace('\n', ' ')
    print(f"  Chunk {i}: [{block_type}] {block_name} ({chunk.token_count} tokens)")
    print(f"           \"{preview}...\"\n")

print("Notice: each function/class stays INTACT in its own chunk!")
print("This is critical — a split function is useless for retrieval.")

In [ ]:
# === Compare chunk counts across strategies ===

# Use the same document with all three strategies
test_doc = loader.load_text(sample_text * 3, source="comparison-test")  # Repeat for more chunks

fixed = FixedSizeChunker(chunk_size=128, overlap=20)
semantic = SemanticChunker(max_chunk_size=256)
code = CodeAwareChunker(max_chunk_size=256)

fixed_result = fixed.chunk(test_doc)
semantic_result = semantic.chunk(test_doc)
code_result = code.chunk(test_doc)

print("=== Strategy Comparison (same input) ===")
print(f"  Input size: ~{test_doc.token_estimate} tokens\n")
print(f"  {'Strategy':<20} {'Chunks':<10} {'Avg Tokens':<12} {'Min':<8} {'Max':<8}")
print(f"  {'─'*58}")

for name, chunks in [("Fixed-Size", fixed_result), ("Semantic", semantic_result), ("Code-Aware", code_result)]:
    if chunks:
        avg = sum(c.token_count for c in chunks) / len(chunks)
        mn = min(c.token_count for c in chunks)
        mx = max(c.token_count for c in chunks)
        print(f"  {name:<20} {len(chunks):<10} {avg:<12.1f} {mn:<8} {mx:<8}")

print("\nKey insight: Fixed-size produces uniform chunks; Semantic produces")
print("variable-size chunks that respect content boundaries.")

---
## 4. Embeddings — Turning Text into Vectors

This is where the magic of RAG happens. Embeddings convert human-readable text into **dense numerical vectors** in a high-dimensional space, where **semantic similarity maps to geometric proximity**.

### What ARE embeddings?

An embedding is a fixed-length vector (array of numbers) that represents the "meaning" of a piece of text. Imagine a 3D room where:
- The X-axis represents "technical vs. casual"
- The Y-axis represents "positive vs. negative"
- The Z-axis represents "code vs. prose"

"Great auth implementation!" would land in the **technical + positive + code** corner.
"Terrible weather today" would be in the **casual + negative + prose** corner.
They're far apart — matching our intuition that they're unrelated.

Real embeddings use **384+ dimensions** instead of 3, capturing nuances humans can't easily name.

### Cosine Similarity

To measure how similar two vectors are, we use **cosine similarity** — it measures the angle between vectors, ignoring magnitude:

```
cos(A, B) = (A . B) / (||A|| * ||B||)
```

- **1.0** = identical direction (same meaning)
- **0.0** = orthogonal (unrelated)
- **-1.0** = opposite (rare in text embeddings)

Why cosine over Euclidean? A 50-token chunk and 500-token chunk about the same topic should be "similar" even though the longer one has a larger-magnitude vector. Cosine ignores magnitude.

In [ ]:
# === Create an embedder and embed some sentences ===

embedder = HashEmbedder(target_dimensions=384)
print(f"Embedder: {embedder.model_name}")
print(f"Output dimensions: {embedder.dimensions}\n")

# Two SIMILAR sentences (about the same topic)
text_a = "authentication with JWT tokens"
text_b = "JWT token-based auth system"

# Two DIFFERENT sentences (unrelated topics)
text_c = "today's weather forecast is sunny"
text_d = "the stock market crashed yesterday"

vec_a = embedder.embed(text_a)
vec_b = embedder.embed(text_b)
vec_c = embedder.embed(text_c)
vec_d = embedder.embed(text_d)

print(f"Vector shape: {vec_a.shape}")
print(f"Vector dtype: {vec_a.dtype}")
print(f"L2 norm (should be ~1.0 for normalized): {np.linalg.norm(vec_a):.4f}")
print(f"\nFirst 10 dimensions of '{text_a}':")
print(f"  {vec_a[:10].round(4)}")

In [ ]:
# === Compute cosine similarity ===
# Similar texts should score HIGH, different texts should score LOW

sim_ab = CosineDistance.similarity(vec_a, vec_b)  # Similar topics
sim_ac = CosineDistance.similarity(vec_a, vec_c)  # Different topics
sim_cd = CosineDistance.similarity(vec_c, vec_d)  # Different topics
sim_aa = CosineDistance.similarity(vec_a, vec_a)  # Identical (should be 1.0)

print("=== Cosine Similarity Results ===")
print(f"  '{text_a}'")
print(f"  vs '{text_b}'")
print(f"  Similarity: {sim_ab:.4f}  ← SIMILAR topics (should be high)\n")

print(f"  '{text_a}'")
print(f"  vs '{text_c}'")
print(f"  Similarity: {sim_ac:.4f}  ← DIFFERENT topics (should be low)\n")

print(f"  '{text_c}'")
print(f"  vs '{text_d}'")
print(f"  Similarity: {sim_cd:.4f}  ← DIFFERENT topics (should be low)\n")

print(f"  Identical text vs itself: {sim_aa:.4f}  ← Should be 1.0")
print(f"\nThe hash embedder uses character n-grams, so texts sharing subwords")
print(f"(like 'auth' and 'authentication') get higher similarity.")

In [ ]:
# === Similarity Matrix — visualize relationships between 5 texts ===

texts = [
    "JWT authentication and token validation",
    "user login and auth credentials",
    "vector database for similarity search",
    "embedding models convert text to vectors",
    "machine learning model training",
]

# Embed all texts
vectors = embedder.embed_batch(texts)

# Compute pairwise similarity matrix
n = len(texts)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i][j] = CosineDistance.similarity(vectors[i], vectors[j])

# Print as formatted table
print("=== Embedding Similarity Matrix ===")
print("(Higher = more similar. Diagonal = 1.0 = identical)\n")

# Short labels
labels = ["JWT auth", "login/creds", "vector DB", "embeddings", "ML training"]

# Header
header = f"{'':>14}" + "".join(f"{l:>13}" for l in labels)
print(header)
print("  " + "─" * (14 + 13 * n))

for i in range(n):
    row = f"  {labels[i]:>12} │"
    for j in range(n):
        score = sim_matrix[i][j]
        # Highlight high similarity
        if i == j:
            row += f"{'1.000':>12} "
        elif score > 0.4:
            row += f"{score:>11.3f}* "  # Star for notable similarity
        else:
            row += f"{score:>12.3f} "
    print(row)

print("\n  * = notable similarity (> 0.4)")
print("\nNotice: auth-related texts cluster together, and")
print("embedding/vector texts cluster together — even though")
print("they use different words!")

---
## 5. Vector Store — The Memory of Your RAG System

A vector store is where your embedded chunks live. When a query comes in, you embed it, then search the vector store for the most similar stored vectors.

### How similarity search works (it's simpler than you think!)

1. Store N vectors in a matrix V of shape (N, d)
2. When a query arrives, embed it to a vector q of shape (d,)
3. Compute similarity scores: `scores = V @ q` (matrix-vector product)
4. Sort by score descending and take the top K

That's it. The entire "vector database" concept boils down to this. Everything else (indexing, sharding, persistence) is optimization.

Our `InMemoryVectorStore` uses brute-force search with numpy. This is perfectly fine for up to ~100K vectors (<100ms per query). Beyond that, you'd use approximate nearest neighbor (ANN) algorithms like HNSW.

In [ ]:
# === Build a vector store and search it ===

store = InMemoryVectorStore()

# Sample knowledge base about different topics
knowledge = [
    ("auth-1", "JWT tokens are used for stateless authentication. Each token contains a header, payload, and signature. The signature is verified using RS256 or HS256 algorithms."),
    ("auth-2", "OAuth 2.0 provides delegated authorization. Users grant third-party apps limited access without sharing credentials. Common flows include Authorization Code and PKCE."),
    ("rag-1", "RAG pipelines retrieve relevant documents before generating an answer. This grounds the LLM in factual data and reduces hallucination dramatically."),
    ("rag-2", "Vector embeddings represent text as dense numerical vectors. Cosine similarity between vectors measures semantic relatedness, enabling meaning-based search."),
    ("deploy-1", "Kubernetes orchestrates containerized applications across a cluster. Pods are the smallest deployable unit, managed by Deployments and exposed via Services."),
    ("deploy-2", "Docker containers package an application with all its dependencies into a portable unit. Dockerfiles define the build steps for creating container images."),
    ("db-1", "PostgreSQL is an advanced open-source relational database. It supports JSONB columns, full-text search, and ACID transactions for data integrity."),
    ("db-2", "Redis is an in-memory data store used for caching, session management, and real-time analytics. It supports data structures like strings, hashes, and sorted sets."),
]

# Embed and store each document
for doc_id, content in knowledge:
    vector = embedder.embed(content)
    record = VectorRecord(
        id=doc_id,
        vector=vector,
        content=content,
        metadata={"topic": doc_id.split("-")[0]},
        document_id=doc_id,
    )
    store.add(record)

print(f"Vector store populated: {store.count()} records\n")

# Search!
query = "How does token-based authentication work?"
query_vector = embedder.embed(query)
results = store.search(query_vector, top_k=3)

print(f"Query: \"{query}\"\n")
print(f"Top 3 results:")
for r in results:
    print(f"  [{r.rank}] Score: {r.score:.4f} | ID: {r.record.id}")
    print(f"      {r.record.content[:100]}...\n")

In [ ]:
# === How top_k and min_score affect results ===

query = "How do containers work in production?"
query_vector = embedder.embed(query)

print(f"Query: \"{query}\"\n")

# top_k=5, no min_score filter
results_all = store.search(query_vector, top_k=5, min_score=0.0)
print(f"=== top_k=5, min_score=0.0 (return everything) ===")
for r in results_all:
    print(f"  [{r.rank}] {r.score:.4f} | {r.record.id:<10} | {r.record.content[:60]}...")

# Same query but with min_score threshold
results_filtered = store.search(query_vector, top_k=5, min_score=0.3)
print(f"\n=== top_k=5, min_score=0.3 (only confident matches) ===")
for r in results_filtered:
    print(f"  [{r.rank}] {r.score:.4f} | {r.record.id:<10} | {r.record.content[:60]}...")
if not results_filtered:
    print("  (No results above threshold)")

# With metadata filtering
results_topic = store.search(query_vector, top_k=3, filters={"topic": "deploy"})
print(f"\n=== top_k=3, filter: topic='deploy' (scoped search) ===")
for r in results_topic:
    print(f"  [{r.rank}] {r.score:.4f} | {r.record.id:<10} | {r.record.content[:60]}...")

print(f"\n--- Collection Stats ---")
stats = store.get_collection_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

---
## 6. Full RAG Pipeline — Putting It All Together

Now that you understand each component, let's wire them together using `RAGPipeline`. The pipeline handles the full lifecycle: configure components, ingest documents, and query.

### The end-to-end flow

```
Ingest:  Document → Chunker → Embedder → Vector Store
Query:   Question → Embedder → Vector Search → Reranker → Context Assembly
```

Without a pipeline, you'd write verbose code manually wiring each stage. The pipeline wraps everything into two methods: `ingest_document()` and `query()`.

In [ ]:
# === Create a RAG Pipeline with configuration ===

config = RAGConfig(
    chunk_strategy="hybrid",         # Auto-route: Python→code-aware, Markdown→semantic, else→fixed
    chunk_size=256,                  # Max tokens per chunk
    chunk_overlap=30,                # Token overlap for fixed-size chunks
    embedding_model="hash",          # HashEmbedder (no training needed)
    embedding_dimensions=384,        # Vector dimensionality
    retrieval_strategy="hybrid",     # Combine semantic + keyword search
    retrieval_top_k=5,              # Retrieve top 5 chunks
    rerank=True,                     # Apply cross-encoder-style reranking
    rerank_top_k=3,                 # Keep top 3 after reranking
    collection_name="tutorial",
)

pipeline = RAGPipeline(config)
print(f"Pipeline created with config:")
print(f"  Chunking:   {config.chunk_strategy} (max {config.chunk_size} tokens)")
print(f"  Embedding:  {config.embedding_model} ({config.embedding_dimensions}d)")
print(f"  Retrieval:  {config.retrieval_strategy}")
print(f"  Reranking:  {'enabled' if config.rerank else 'disabled'}")

In [ ]:
# === Ingest sample documents about different topics ===

documents = [
    ("Authentication uses JWT tokens with RS256 signing. Each token contains a header "
     "with the algorithm type, a payload with user claims (user_id, roles, expiry), "
     "and a cryptographic signature. The server validates tokens by checking the signature "
     "against the public key and verifying the expiry timestamp. Refresh tokens are issued "
     "alongside access tokens to allow seamless re-authentication without requiring the "
     "user to log in again. Token rotation prevents replay attacks.",
     {"topic": "auth", "component": "security"}),
    
    ("The agent framework uses a BaseAgent class that provides standard lifecycle hooks: "
     "initialize, plan, execute, and report. Each agent specializes in a domain like "
     "code review, security scanning, or documentation generation. Agents communicate "
     "through an A2A (Agent-to-Agent) protocol using structured messages. The Mission "
     "Control layer orchestrates multi-agent workflows, routing tasks to the most "
     "capable agent based on trust scores and specialization.",
     {"topic": "agents", "component": "core"}),
    
    ("The RAG pipeline processes documents through five stages: loading, chunking, "
     "embedding, storage, and retrieval. Chunking strategies include fixed-size (token "
     "windows with overlap), semantic (paragraph/heading boundaries), and code-aware "
     "(function/class boundaries). The embedder converts chunks to 384-dimensional "
     "vectors using feature hashing. The vector store enables cosine similarity search "
     "for finding the most relevant chunks for a given query.",
     {"topic": "rag", "component": "knowledge"}),
    
    ("Deployment uses Kubernetes with Helm charts for orchestration. The platform runs "
     "as a set of microservices: API gateway, agent runtime, vector store, and dashboard. "
     "Horizontal pod autoscaling adjusts replicas based on CPU and request latency. "
     "Secrets are managed through Kubernetes Secrets and mounted as environment variables. "
     "CI/CD pipelines use GitHub Actions for automated testing, building container images, "
     "and deploying to staging and production clusters.",
     {"topic": "deployment", "component": "infra"}),
    
    ("The FinOps module tracks resource consumption and cost allocation per tenant. "
     "It meters API calls, token usage (input and output tokens for LLM calls), "
     "compute time, and storage consumption. Usage data is aggregated into daily "
     "and monthly reports. Budget alerts trigger when a tenant exceeds configured "
     "thresholds. Cost optimization recommendations include right-sizing models, "
     "caching frequent queries, and batching embeddings.",
     {"topic": "finops", "component": "billing"}),
]

print("Ingesting 5 documents into the pipeline...\n")
for i, (content, metadata) in enumerate(documents):
    doc = loader.load_text(content, metadata=metadata, source=f"doc-{metadata['topic']}")
    result = pipeline.ingest_document(doc)
    print(f"  [{i+1}] topic={metadata['topic']:<12} → {result.chunks_created} chunks, "
          f"{result.tokens_total} tokens, {result.time_ms:.1f}ms")

print(f"\nTotal vectors in store: {pipeline.vector_store.count(config.collection_name)}")

In [ ]:
# === Query the pipeline with different questions ===

queries = [
    "How does authentication work?",
    "What agents are available in the platform?",
    "How is the RAG pipeline structured?",
    "How do you deploy the application?",
    "How does cost tracking work?",
]

for query in queries:
    result = pipeline.query(query, top_k=3)
    print(f"\nQ: \"{query}\"")
    print(f"   Search: {result.search_time_ms:.1f}ms | "
          f"Chunks found: {len(result.chunks)} | "
          f"Context tokens: {result.tokens_in_context}")
    
    for i, (chunk, score) in enumerate(zip(result.chunks, result.scores)):
        source = chunk.metadata.get('source', 'unknown')
        topic = chunk.metadata.get('topic', '?')
        preview = chunk.content[:80].replace('\n', ' ')
        print(f"   [{i+1}] score={score:.4f} topic={topic:<12} \"{preview}...\"")

In [ ]:
# === Pipeline statistics ===

import json
stats = pipeline.get_stats()
print("=== Pipeline Statistics ===")
print(json.dumps(stats, indent=2, default=str))

---
## 7. Hybrid Retrieval — Best of Both Worlds

Semantic search and keyword search have **complementary failure modes**:

| Strategy | Strengths | Weaknesses |
|---|---|---|
| **Semantic** | Handles synonyms, paraphrases, conceptual queries | Misses exact strings, error codes, variable names |
| **Keyword (BM25)** | Great for exact matches, error codes, jargon | Fails on synonyms ("car" won't find "automobile") |
| **Hybrid** | Covers both blind spots | Slightly more computation |

### Reciprocal Rank Fusion (RRF)

How do you merge two ranked lists? You can't just average scores because cosine similarity is in [0,1] while BM25 is in [0, infinity] — they're not comparable.

**RRF** solves this by only using **ranks**, which are always comparable:

```
RRF(doc) = sum(1 / (k + rank_i) for each list where doc appears)
```

Documents found by BOTH retrievers get higher RRF scores — being relevant in multiple ways is a strong signal.

In [ ]:
# === Compare retrieval strategies on the same query ===

# Create all three retriever types sharing the same embedder + store
shared_embedder = pipeline.embedder
shared_store = pipeline.vector_store

semantic_retriever = SemanticRetriever(shared_embedder, shared_store)
keyword_retriever = KeywordRetriever(shared_store)
hybrid_retriever = HybridRetriever(shared_embedder, shared_store)

test_query = "JWT token validation and authentication"
collection = config.collection_name

print(f"Query: \"{test_query}\"\n")

# Run each retriever
for name, retriever in [("Semantic", semantic_retriever), 
                        ("Keyword (BM25)", keyword_retriever),
                        ("Hybrid (RRF)", hybrid_retriever)]:
    result = retriever.retrieve(test_query, top_k=3, collection=collection)
    print(f"=== {name} ({result.search_time_ms:.1f}ms) ===")
    for i, (chunk, score) in enumerate(zip(result.chunks, result.scores)):
        topic = chunk.metadata.get('topic', '?')
        preview = chunk.content[:70].replace('\n', ' ')
        print(f"  [{i+1}] {score:.4f} | topic={topic:<12} | {preview}...")
    print()

In [ ]:
# === Demonstrate where keyword search wins ===
# Exact terms like function names or error codes are keyword territory

exact_query = "BaseAgent framework lifecycle hooks"
print(f"Query: \"{exact_query}\"")
print(f"(This query uses exact terms from the codebase)\n")

for name, retriever in [("Semantic", semantic_retriever), 
                        ("Keyword", keyword_retriever),
                        ("Hybrid", hybrid_retriever)]:
    result = retriever.retrieve(exact_query, top_k=2, collection=collection)
    print(f"  {name:>10}: ", end="")
    if result.chunks:
        top_topic = result.chunks[0].metadata.get('topic', '?')
        print(f"top match = '{top_topic}' (score: {result.scores[0]:.4f})")
    else:
        print("no results")

print("\nHybrid retrieval combines the strengths of both approaches,")
print("so it performs well regardless of whether the query is")
print("conceptual (semantic) or precise (keyword).")

---
## 8. Evaluation — Measuring Retrieval Quality

You can't improve what you can't measure. The `RAGEvaluator` computes standard Information Retrieval metrics:

| Metric | What It Measures | Formula |
|---|---|---|
| **Precision@K** | Of K retrieved chunks, how many were relevant? | relevant_in_top_k / K |
| **Recall@K** | Of all relevant chunks, how many did we find? | found_relevant / total_relevant |
| **MRR** | How quickly do we find the first relevant result? | 1 / rank_of_first_relevant |
| **nDCG@K** | How good is the ranking order? | DCG / ideal_DCG |
| **Faithfulness** | Is the context grounded in source material? | grounded_tokens / total_tokens |

### Reading the results
- **Precision > 0.7**: Good — few irrelevant results (low noise)
- **Recall > 0.8**: Good — finding most relevant chunks
- **MRR > 0.5**: Good — relevant results near the top
- **nDCG > 0.7**: Good — ranking quality is solid

In [ ]:
# === Define evaluation test cases ===
# Each case has: a query, expected chunk identifiers, and optionally an expected answer.

eval_cases = [
    EvalCase(
        query="How does JWT authentication work?",
        expected_chunks=["auth"],  # Should find auth-related chunks
        expected_answer="JWT tokens with RS256 signing, payload contains user claims",
    ),
    EvalCase(
        query="What is the agent framework?",
        expected_chunks=["agents"],  # Should find agent-related chunks
        expected_answer="BaseAgent class with lifecycle hooks: initialize, plan, execute, report",
    ),
    EvalCase(
        query="How does the RAG pipeline process documents?",
        expected_chunks=["rag"],
        expected_answer="Five stages: loading, chunking, embedding, storage, retrieval",
    ),
    EvalCase(
        query="How is the application deployed?",
        expected_chunks=["deployment"],
        expected_answer="Kubernetes with Helm charts, microservices architecture",
    ),
    EvalCase(
        query="How does cost metering work?",
        expected_chunks=["finops"],
        expected_answer="Meters API calls, token usage, compute time per tenant",
    ),
]

print(f"Defined {len(eval_cases)} evaluation cases:")
for i, case in enumerate(eval_cases):
    print(f"  [{i+1}] Q: \"{case.query}\"")
    print(f"       Expected: {case.expected_chunks}")

In [ ]:
# === Run the evaluation ===

evaluator = RAGEvaluator()
results = evaluator.evaluate_retrieval(pipeline, eval_cases, top_k=3)

print("=== RAG Evaluation Results ===")
print(f"  Number of cases: {results['num_cases']}\n")

print("  Retrieval Metrics:")
print(f"    Precision@3:  {results['precision_at_k']:.4f}  "
      f"{'(Good!)' if results['precision_at_k'] > 0.7 else '(Needs improvement)'}")
print(f"    Recall@3:     {results['recall_at_k']:.4f}  "
      f"{'(Good!)' if results['recall_at_k'] > 0.7 else '(Needs improvement)'}")
print(f"    MRR:          {results['mrr']:.4f}  "
      f"{'(Good!)' if results['mrr'] > 0.5 else '(Needs improvement)'}")
print(f"    nDCG@3:       {results['ndcg_at_k']:.4f}  "
      f"{'(Good!)' if results['ndcg_at_k'] > 0.7 else '(Needs improvement)'}")

print(f"\n  Answer Quality:")
print(f"    Faithfulness:  {results['faithfulness']:.4f}  "
      f"{'(Good!)' if results['faithfulness'] > 0.8 else '(Could improve)'}")

# Also generate the full report
print(f"\n{'='*60}")
report = evaluator.generate_report(results, eval_cases)
print(report)

---
## 9. Ingesting the SevaForge Codebase — Real-World RAG

Let's go beyond toy examples and ingest actual SevaForge source files. This demonstrates real-world RAG ingestion where documents are real code with classes, functions, and module docstrings.

This is the **dog-fooding** test: can SevaForge's RAG pipeline understand its own codebase?

In [ ]:
# === Ingest actual SevaForge RAG source files ===

codebase_config = RAGConfig(
    chunk_strategy="hybrid",          # Code-aware for Python, semantic for Markdown
    embedding_model="hash",
    retrieval_strategy="hybrid",
    rerank=True,
    rerank_top_k=3,
    collection_name="sevaforge_code",
)

code_pipeline = RAGPipeline(codebase_config)

# Ingest the RAG module source files
import os
rag_dir = os.path.abspath("../src/sevaforge/rag")
print(f"Ingesting from: {rag_dir}\n")

ingest_results = code_pipeline.ingest_directory(rag_dir, extensions=[".py"])

total_chunks = sum(r.chunks_created for r in ingest_results)
total_tokens = sum(r.tokens_total for r in ingest_results)
total_time = sum(r.time_ms for r in ingest_results)

print(f"=== Ingestion Summary ===")
print(f"  Files ingested:  {len(ingest_results)}")
print(f"  Chunks created:  {total_chunks}")
print(f"  Total tokens:    {total_tokens}")
print(f"  Total time:      {total_time:.1f}ms")
print(f"  Throughput:      {total_chunks / max(total_time, 1) * 1000:.0f} chunks/sec\n")

for r in ingest_results:
    print(f"  {r.document_id[:16]}... → {r.chunks_created:>3} chunks, {r.tokens_total:>5} tokens")

In [ ]:
# === Query your own codebase ===

codebase_queries = [
    "How does the chunking strategy work?",
    "What embedding models are available?",
    "How does cosine similarity search work in the vector store?",
    "What is Reciprocal Rank Fusion?",
    "How does the RAG evaluation compute nDCG?",
]

print("=== Querying SevaForge's Own Codebase ===")
for query in codebase_queries:
    result = code_pipeline.query(query, top_k=2)
    print(f"\nQ: \"{query}\"")
    print(f"   ({result.search_time_ms:.1f}ms, {len(result.chunks)} chunks, "
          f"{result.tokens_in_context} context tokens)")
    
    for i, (chunk, score) in enumerate(zip(result.chunks, result.scores)):
        block = chunk.metadata.get('block_name', '')
        btype = chunk.metadata.get('block_type', '')
        label = f"{btype}::{block}" if block else chunk.metadata.get('source', '?')[-30:]
        # Get a clean preview
        lines = chunk.content.strip().split('\n')
        preview = lines[0][:80] if lines else ''
        print(f"   [{i+1}] score={score:.4f} | {label}")
        print(f"       {preview}")

---
## 10. Multi-Tenant RAG Service

In production, the RAG pipeline runs as an API service. SevaForge exposes RESTful endpoints via FastAPI for document ingestion, querying, and pipeline management.

### Key endpoints

| Method | Endpoint | Purpose |
|---|---|---|
| POST | `/rag/ingest` | Ingest text content |
| POST | `/rag/ingest/file` | Ingest from file path |
| POST | `/rag/ingest/directory` | Ingest entire directory |
| POST | `/rag/query` | Query the pipeline |
| GET | `/rag/collections` | List collections |
| GET | `/rag/collections/{name}/stats` | Collection statistics |
| DELETE | `/rag/collections/{name}` | Clear a collection |
| POST | `/rag/evaluate` | Run evaluation suite |

Each tenant gets an isolated pipeline instance with its own vector store, ensuring full data separation.

Here is how you would interact with the service programmatically:

In [ ]:
# === Simulated API interaction (demonstrates the request/response format) ===
# In production, you'd use httpx or requests against the running FastAPI server.
# Here we simulate the flow to show the API contract.

# Simulate: POST /rag/ingest
ingest_request = {
    "content": "SevaForge is a multi-agent AI platform that orchestrates "
               "specialized agents for software development tasks.",
    "source": "api-docs",
    "doc_type": "text",
    "metadata": {"version": "1.0", "category": "overview"},
    "collection": "default",
    "tenant_id": "demo-tenant",
}
print("POST /rag/ingest")
print(f"  Request body: {json.dumps(ingest_request, indent=2)}\n")

# Simulate: POST /rag/query
query_request = {
    "question": "What is SevaForge?",
    "top_k": 3,
    "collection": "default",
    "tenant_id": "demo-tenant",
}
print("POST /rag/query")
print(f"  Request body: {json.dumps(query_request, indent=2)}\n")

# Simulate response structure
query_response = {
    "answer_context": "--- Source: api-docs ---\nSevaForge is a multi-agent AI platform...",
    "num_chunks": 1,
    "search_time_ms": 12.5,
    "tokens_in_context": 42,
    "total_chunks_searched": 150,
}
print("Response (200 OK):")
print(f"  {json.dumps(query_response, indent=2)}")

print("\nTo run the actual API server:")
print("  uvicorn sevaforge.api.app:app --reload --port 8000")
print("  Then visit http://localhost:8000/docs for Swagger UI")

---
## 11. Next Steps — From Tutorial to Production

Congratulations! You now understand the full RAG pipeline from first principles. Here's how to take this to production:

### Upgrade the Embedder

Our `HashEmbedder` is great for prototyping (zero setup, deterministic), but it's purely lexical. For production, upgrade to neural embeddings:

```python
# Just pip install sentence-transformers, then:
from sevaforge.rag import SentenceTransformerEmbedder
embedder = SentenceTransformerEmbedder(model_name_or_path="all-MiniLM-L6-v2")
# Now "car" and "automobile" will have similar embeddings!
```

### Upgrade the Vector Store

Our `InMemoryVectorStore` is fine for <100K vectors. For production:

| Scale | Solution | Why |
|---|---|---|
| <100K vectors | InMemoryVectorStore | Simple, fast, zero dependencies |
| 100K-1M | ChromaDB (local) | Persistence, HNSW indexing |
| 1M-100M | Pinecone / Weaviate / Qdrant | Managed, sharded, production-grade |
| 100M+ | Google Vertex AI / Pinecone Serverless | Cloud-scale, tiered storage |

### Add LLM Generation

RAG retrieval gives you the context. The final step is passing it to an LLM:

```python
result = pipeline.query("How does authentication work?")

prompt = f"""Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say "I don't know."

Context:
{result.answer_context}

Question: How does authentication work?
Answer:"""

# Pass to Claude, GPT-4, or any LLM
# answer = llm.generate(prompt)
```

### Fine-Tune Embeddings

For best results on your specific domain, fine-tune the embedding model on your data using contrastive learning. This teaches the model that domain-specific terms (like "SevaForge" and "agent framework") are related concepts.

### Production Checklist

- [ ] Switch to neural embeddings (sentence-transformers or OpenAI)
- [ ] Use a persistent vector store (ChromaDB, Pinecone)
- [ ] Set up incremental ingestion (only re-process changed files)
- [ ] Add metadata filtering for scoped searches
- [ ] Implement evaluation CI (run eval suite on every pipeline change)
- [ ] Monitor retrieval latency and quality metrics in production
- [ ] Add caching for frequent queries (SevaForge's SemanticCache)
- [ ] Implement token budgeting for LLM context windows

---

## Summary

In this notebook, you learned:

1. **What RAG is** and why it matters (grounding LLMs in real data)
2. **Document Loading** — reading files and extracting structural metadata
3. **Chunking** — the most critical decision: fixed-size, semantic, and code-aware strategies
4. **Embeddings** — converting text to vectors where similar meanings are geometrically close
5. **Vector Stores** — storing and searching vectors with cosine similarity
6. **The Full Pipeline** — wiring everything together with `RAGPipeline`
7. **Hybrid Retrieval** — combining semantic and keyword search with Reciprocal Rank Fusion
8. **Evaluation** — measuring retrieval quality with Precision, Recall, MRR, and nDCG
9. **Real-world ingestion** — SevaForge searching its own codebase
10. **Production path** — upgrading from prototype to production-grade RAG

Everything in this tutorial was built from first principles using only `numpy` and Python's standard library. You now understand what frameworks like LangChain and vector databases like ChromaDB are doing under the hood.

**Happy building!**